## CELL 1: SETUP & CONFIGURATION

In [0]:
import logging
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, current_timestamp, lit, rand, when
from pyspark.sql.types import IntegerType

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_param(param_name: str, default_value: str) -> str:
    try:
        return dbutils.widgets.get(param_name)
    except Exception:
        return default_value

# Pull from orchestrator (variables.yml) or fallback to local defaults
CATALOG = get_param("catalog_name", "main")
SCHEMA = get_param("schema_name", "fixkart_prd")

# Dynamic Table Paths
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_bookings"
DIM_TECH_TABLE = f"{CATALOG}.{SCHEMA}.dim_technician"
DIM_CUST_TABLE = f"{CATALOG}.{SCHEMA}.dim_customer"
FACT_TABLE = f"{CATALOG}.{SCHEMA}.fact_work_orders"
VIEW_NAME = f"{CATALOG}.{SCHEMA}.semantic_executive_kpis"

logger.info(f"Initialized Gold Pipeline | Schema: {CATALOG}.{SCHEMA}")

## CELL 2: STAR SCHEMA GENERATION

In [0]:
def build_dimensions(spark: SparkSession, source_table: str, dim_tech: str, dim_cust: str) -> None:
    """Extracts and overwrites dimension tables from the Silver layer."""
    logger.info("Building dimension tables...")
    try:
        df_silver = spark.table(source_table)
        
        # Technician Dimension
        df_dim_tech = df_silver.select("technician_id", "city").dropDuplicates(["technician_id"])
        df_dim_tech.write.format("delta").mode("overwrite").saveAsTable(dim_tech)
        
        # Customer Dimension
        df_dim_cust = df_silver.select("customer_id").dropDuplicates(["customer_id"])
        df_dim_cust.write.format("delta").mode("overwrite").saveAsTable(dim_cust)
        
        logger.info(f"Dimensions {dim_tech} and {dim_cust} successfully rebuilt.")
    except Exception as e:
        logger.error(f"Error building dimensions: {str(e)}")
        raise

def build_fact_table(spark: SparkSession, source_table: str, fact_table: str) -> None:
    """Builds the central fact table, supplementing missing metrics for the semantic layer."""
    logger.info("Building fact table...")
    try:
        df_silver = spark.table(source_table)
        
        df_fact = (
            df_silver.select(
                "work_order_id", "customer_id", "technician_id", "booking_date", 
                "status", "failure_reason", "reschedule_count"
            )
            # Handle missing fields from mock data to support the downstream semantic view
            .withColumn("channel", lit("Mobile App"))
            .withColumn(
                "delay_minutes", 
                when(col("status") == "COMPLETED", (rand() * 45).cast(IntegerType())).otherwise(lit(0))
            )
            .withColumn("last_updated", current_timestamp())
        )
        
        df_fact.write.format("delta").mode("overwrite").saveAsTable(fact_table)
        logger.info(f"Fact table {fact_table} successfully rebuilt.")
    except Exception as e:
        logger.error(f"Error building fact table: {str(e)}")
        raise

## CELL 3: UNITY CATALOG CONSTRAINTS

In [0]:
def build_semantic_layer(spark: SparkSession, view_name: str, fact_table: str, dim_tech: str) -> None:
    """Deploys the final Business Intelligence views."""
    logger.info(f"Deploying Semantic View: {view_name}...")
    
    semantic_sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT 
        t.city,
        COUNT(f.work_order_id) AS total_dispatches,
        
        -- PM Metric 1: Capacity Leakage (₹350 cost per failed visit)
        SUM(CASE WHEN f.status = 'FAILED' THEN 1 ELSE 0 END) AS failed_visits,
        SUM(CASE WHEN f.status = 'FAILED' THEN 1 ELSE 0 END) * 350 AS revenue_leakage_inr,
        
        -- PM Metric 2: Technician Utilization
        ROUND((SUM(CASE WHEN f.status = 'COMPLETED' THEN 1 ELSE 0 END) / COUNT(f.work_order_id)) * 100, 2) AS utilization_rate_pct,
        
        -- PM Metric 3: Friction Index (High reschedules leading to failure)
        SUM(CASE WHEN f.status = 'FAILED' AND f.reschedule_count > 0 THEN 1 ELSE 0 END) AS high_friction_failures,
        
        -- PM Metric 4: Average SLA Delay
        ROUND(AVG(f.delay_minutes), 1) AS avg_delay_minutes
        
    FROM {fact_table} f
    JOIN {dim_tech} t ON f.technician_id = t.technician_id
    GROUP BY t.city
    """
    try:
        spark.sql(semantic_sql)
        logger.info("Semantic view successfully compiled.")
    except Exception as e:
        logger.error(f"Failed to create semantic view: {str(e)}")
        raise

## Cell 4: SEMANTIC LAYER (VIEWS)

In [0]:
if 'spark' in locals() or 'spark' in globals():
    try:
        logger.info("--- Starting Gold Processing Pipeline ---")
        
        build_dimensions(spark, SILVER_TABLE, DIM_TECH_TABLE, DIM_CUST_TABLE)
        build_fact_table(spark, SILVER_TABLE, FACT_TABLE)
        build_semantic_layer(spark, VIEW_NAME, FACT_TABLE, DIM_TECH_TABLE)
        
        logger.info("--- Gold Pipeline Completed Successfully ---")
        
    except Exception as main_e:
        logger.error("Gold Pipeline execution aborted due to an error.")
else:
    logger.warning("SparkSession not found.")

## CELL 5: ORCHESTRATION & EXECUTION

In [0]:
# Verify the semantic view output
display(spark.table(VIEW_NAME))